# Sentiment Classifier — Twitter-RoBERTa Fine-tuning
### Sunlytics CRS — M3 Sentiment Component

Fine-tunes **`cardiffnlp/twitter-roberta-base-sentiment-latest`** on the preprocessed SemEval 2017 tweet sentiment data.

**Why this model:**
- Pre-trained on 124 million tweets (tweet-native vocabulary)
- Already fine-tuned for 3-class tweet sentiment (negative / neutral / positive)
- Fine-tuning further on our preprocessed data adapts it to our exact class distribution
- Expected accuracy: **~80–83%** vs DistilBERT baseline of 73.8%

**Training data:** `tweet_eval_train_clean.csv` — 45,613 rows (preprocessed)  
**Val data:** `tweet_eval_val_clean.csv` — 2,000 rows  
**Test data:** `tweet_eval_test_clean.csv` — 12,241 rows

### Steps before running
1. `Runtime → Change runtime type → T4 GPU` → Save
2. Run **Cell 1 only** first — installs packages and restarts kernel
3. After restart, run all remaining cells from Cell 2 onward

In [ ]:
# ── CELL 1 — Install packages  (run first, alone) ─────────────────────────────
!pip install "transformers>=4.40.0" "datasets==3.2.0" accelerate scikit-learn pandas -q
print('Packages installed. Restarting kernel...')
import os
os.kill(os.getpid(), 9)

In [ ]:
# ── CELL 2 — Check GPU ────────────────────────────────────────────────────────
import torch
print('GPU available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device        :', torch.cuda.get_device_name(0))
    print('Memory (GB)   :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── CELL 3 — Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_PATH = '/content/drive/MyDrive/sentiment_classifier_roberta'
os.makedirs(SAVE_PATH, exist_ok=True)
print('Model will be saved to:', SAVE_PATH)

In [ ]:
# ── CELL 4 — Upload preprocessed CSV files ────────────────────────────────────
# Upload from sentiment_preprocessed_data_set/ on your local machine:
#   tweet_eval_train_clean.csv
#   tweet_eval_val_clean.csv
#   tweet_eval_test_clean.csv

from google.colab import files
import shutil

os.makedirs('data', exist_ok=True)
print('Upload: tweet_eval_train_clean.csv, tweet_eval_val_clean.csv, tweet_eval_test_clean.csv')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'data/{fname}')
    print(f'  Moved {fname} -> data/{fname}')
print('Files in data/:', os.listdir('data/'))

In [ ]:
# ── CELL 5 — Load CSVs and verify ─────────────────────────────────────────────
import pandas as pd
from collections import Counter

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

df_train = pd.read_csv('data/tweet_eval_train_clean.csv')
df_val   = pd.read_csv('data/tweet_eval_val_clean.csv')
df_test  = pd.read_csv('data/tweet_eval_test_clean.csv')

print('=== Dataset sizes ===')
print(f'  Train : {len(df_train):,}')
print(f'  Val   : {len(df_val):,}')
print(f'  Test  : {len(df_test):,}')

for name, df in [('TRAIN', df_train), ('VAL', df_val), ('TEST', df_test)]:
    counts = Counter(df['label_name'].tolist())
    print(f'\n{name} distribution:')
    for lid, lname in ID2LABEL.items():
        pct = counts[lname] / len(df) * 100
        print(f'  {lname:10s}: {counts[lname]:,}  ({pct:.1f}%)')

In [ ]:
# ── CELL 6 — Load Twitter-RoBERTa tokeniser and tokenise ──────────────────────
from datasets import Dataset
from transformers import AutoTokenizer

BASE_MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

print(f'Loading tokeniser: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def df_to_hf(df):
    return Dataset.from_dict({
        'text':  df['text'].tolist(),
        'label': df['label'].tolist(),
    })

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=128,
        padding='max_length',
    )

tokenised = {}
for split, df in [('train', df_train), ('validation', df_val), ('test', df_test)]:
    tok = df_to_hf(df).map(tokenize, batched=True, batch_size=512)
    tok = tok.rename_column('label', 'labels')
    tok.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    tokenised[split] = tok
    print(f'  {split:12s}: {len(tok):,} samples tokenised')

print('\nTokenisation complete.')
print('Input shape (first train sample):', tokenised['train'][0]['input_ids'].shape)

In [ ]:
# ── CELL 7 — Load Twitter-RoBERTa model ───────────────────────────────────────
from transformers import AutoModelForSequenceClassification

print(f'Loading model: {BASE_MODEL}')
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per_class = f1_score(labels, preds, average=None, labels=[0, 1, 2])
    return {
        'accuracy'    : round(accuracy_score(labels, preds), 4),
        'f1_macro'    : round(f1_score(labels, preds, average='macro'), 4),
        'f1_negative' : round(float(f1_per_class[0]), 4),
        'f1_neutral'  : round(float(f1_per_class[1]), 4),
        'f1_positive' : round(float(f1_per_class[2]), 4),
    }

WARMUP_STEPS = 570
NUM_EPOCHS   = 4

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=1e-5,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenised['train'],
    eval_dataset=tokenised['validation'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f'Starting training — no class weights, {NUM_EPOCHS} epochs max, early stopping patience=2...')
trainer.train()

In [ ]:
# ── CELL 9 — Evaluate on held-out test set ────────────────────────────────────
test_results = trainer.evaluate(tokenised['test'])

print('=' * 55)
print('  TEST SET RESULTS  —  Twitter-RoBERTa fine-tuned')
print('=' * 55)
print(f"  Accuracy    : {test_results.get('eval_accuracy', 0):.4f}")
print(f"  F1 Macro    : {test_results.get('eval_f1_macro', 0):.4f}")
print(f"  F1 Negative : {test_results.get('eval_f1_negative', 0):.4f}")
print(f"  F1 Neutral  : {test_results.get('eval_f1_neutral', 0):.4f}")
print(f"  F1 Positive : {test_results.get('eval_f1_positive', 0):.4f}")
print('=' * 55)

In [ ]:
import json

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

metrics = {
    'base_model'             : BASE_MODEL,
    'dataset'                : 'cardiffnlp/tweet_eval sentiment (SemEval 2017) — preprocessed',
    'class_weights'          : 'none',
    'train_size'             : len(df_train),
    'val_size'               : len(df_val),
    'test_size'              : len(df_test),
    'max_epochs'             : NUM_EPOCHS,
    'learning_rate'          : 1e-5,
    'warmup_steps'           : WARMUP_STEPS,
    'early_stopping_patience': 2,
    'test_results'           : {k.replace('eval_', ''): v for k, v in test_results.items()},
}
with open(f'{SAVE_PATH}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to Google Drive:', SAVE_PATH)
print('Files:')
for fname in sorted(os.listdir(SAVE_PATH)):
    size = os.path.getsize(f'{SAVE_PATH}/{fname}')
    print(f'  {fname:40s}  {size/1e6:.1f} MB')

In [ ]:
# ── CELL 11 — Download model as ZIP ──────────────────────────────────────────
import shutil
from google.colab import files

zip_path = '/content/sentiment_classifier_roberta'
shutil.make_archive(zip_path, 'zip', SAVE_PATH)

zip_size = os.path.getsize(f'{zip_path}.zip') / 1e6
print(f'ZIP size: {zip_size:.0f} MB')
print('Starting download...')
files.download(f'{zip_path}.zip')

## After downloading

1. Unzip `sentiment_classifier_roberta.zip`
2. If accuracy ≥ 80% → place the folder at `m3_implementation/memory/models/sentiment_classifier/` (replaces DistilBERT model)
3. `pipeline.py` uses `AutoModelForSequenceClassification` and `AutoTokenizer` — it will load this model automatically

`training_metrics.json` has all scores for thesis comparison table.

---
## Evaluation Analysis (run independently — no retraining needed)
Load the already-trained RoBERTa model from Drive and generate visualisations.  
Run from Cell E1 onward.

In [ ]:
# ── CELL E1 — Setup ───────────────────────────────────────────────────────────
!pip install "transformers>=4.40.0" scikit-learn seaborn pandas -q

from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_PATH = '/content/drive/MyDrive/sentiment_classifier_roberta'
EVAL_PATH  = '/content/drive/MyDrive/sentiment_classifier_roberta/eval_results'
os.makedirs(EVAL_PATH, exist_ok=True)

print('Model path :', MODEL_PATH)
for f in ['config.json', 'model.safetensors', 'tokenizer.json']:
    status = '✓' if os.path.isfile(f'{MODEL_PATH}/{f}') else '✗ MISSING'
    print(f'  {f}: {status}')

In [ ]:
# ── CELL E2 — Upload test CSV + run predictions ───────────────────────────────
import numpy as np
import torch
import pandas as pd
import shutil
from google.colab import files
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ID2LABEL = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

os.makedirs('data', exist_ok=True)
print('Upload: tweet_eval_test_clean.csv')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, f'data/{fname}')

df_test = pd.read_csv('data/tweet_eval_test_clean.csv')
print(f'Test set: {len(df_test):,} rows')

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f'Model loaded — running on {device}')

BATCH = 64
texts, labels_true = df_test['text'].tolist(), df_test['label'].tolist()
all_preds = []

for i in range(0, len(texts), BATCH):
    enc = tokenizer(texts[i:i+BATCH], truncation=True, max_length=128,
                    padding='max_length', return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())

y_pred = np.array(all_preds)
y_true = np.array(labels_true)
print(f'Inference complete — {len(y_pred):,} predictions')

In [ ]:
# ── CELL E3 — Confusion Matrix ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

label_names = ['Negative', 'Neutral', 'Positive']
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'Confusion Matrix — Twitter-RoBERTa on SemEval 2017 Test Set ({len(y_true):,} samples)',
    fontsize=13, fontweight='bold'
)
for ax, data, fmt, title in zip(
    axes, [cm, cm_pct], ['d', '.1f'], ['Raw Counts', 'Row-normalised (%)']
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=ax,
                linewidths=0.5, linecolor='gray', annot_kws={'size': 13})
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# ── CELL E4 — Per-class Precision / Recall / F1 ───────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

label_names = ['Negative', 'Neutral', 'Positive']
report = classification_report(y_true, y_pred, target_names=label_names, output_dict=True)

x, width = np.arange(len(label_names)), 0.25
colors   = ['#4C72B0', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(10, 6))
for i, (metric, color) in enumerate(zip(['precision', 'recall', 'f1-score'], colors)):
    values = [report[l][metric] for l in label_names]
    bars = ax.bar(x + i * width, values, width, label=metric.capitalize(),
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

macro_f1 = report['macro avg']['f1-score']
ax.axhline(y=macro_f1, color='black', linestyle='--', linewidth=1.5,
           label=f'Macro F1 = {macro_f1:.4f}')
ax.set_xlabel('Sentiment Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-class Metrics — Twitter-RoBERTa on SemEval 2017 Test Set',
             fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(label_names, fontsize=12)
ax.set_ylim(0, 1.10)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nFull Classification Report:')
print(classification_report(y_true, y_pred, target_names=label_names))
print('Saved: per_class_metrics.png')

In [ ]:
# ── CELL E5 — Metrics Summary Card ───────────────────────────────────────────
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
import json

acc      = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')
f1_vals  = f1_score(y_true, y_pred, average=None)

fig, ax = plt.subplots(figsize=(8, 5))
ax.axis('off')
fig.patch.set_facecolor('#f8f9fa')
ax.text(0.5, 0.95, 'Twitter-RoBERTa Sentiment Classifier — Final Evaluation',
        ha='center', va='top', fontsize=13, fontweight='bold', transform=ax.transAxes)
ax.text(0.5, 0.87,
        f'Dataset: SemEval 2017 (preprocessed)  |  Test set: {len(y_true):,} samples',
        ha='center', va='top', fontsize=10, color='gray', transform=ax.transAxes)

rows = [
    ('Accuracy',      f'{acc:.4f}',       f'{acc*100:.2f}%'),
    ('Macro F1',      f'{f1_macro:.4f}',  f'{f1_macro*100:.2f}%'),
    ('F1 — Negative', f'{f1_vals[0]:.4f}',''),
    ('F1 — Neutral',  f'{f1_vals[1]:.4f}',''),
    ('F1 — Positive', f'{f1_vals[2]:.4f}',''),
]
table = ax.table(cellText=[[r[0], r[1], r[2]] for r in rows],
                 colLabels=['Metric', 'Score', ''], loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.0)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#343a40')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#e9ecef')
    cell.set_edgecolor('white')

plt.tight_layout()
plt.savefig(f'{EVAL_PATH}/metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()

eval_summary = {
    'model':        'twitter-roberta-base-sentiment-latest (fine-tuned)',
    'dataset':      'cardiffnlp/tweet_eval sentiment (SemEval 2017) — preprocessed',
    'test_samples': int(len(y_true)),
    'accuracy':     round(float(acc), 4),
    'f1_macro':     round(float(f1_macro), 4),
    'f1_negative':  round(float(f1_vals[0]), 4),
    'f1_neutral':   round(float(f1_vals[1]), 4),
    'f1_positive':  round(float(f1_vals[2]), 4),
}
with open(f'{EVAL_PATH}/eval_summary.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print('Saved: metrics_summary.png')
print('Saved: eval_summary.json')
print('\nEvaluation files in:', EVAL_PATH)